# Cleaning DSLD labels into a label set

The raw loader draws records from the dump, the service cleans and flattens
them, and the service saves the rows as a named label set under
`data/dsld/labels/<name>/`. That set is what the label-set loader's
`load_flattened(name)` reads and what a Purpose/Risk run runs over.

The service is renderer v4 (spec 0005): passages are normalised and have
checklist boilerplate cut, so a set saved here is `-r4`. Sets saved under
v1-v3 stay on disk as they were written.

Each cell is one call.

In [ ]:
from engine.repositories.dsld.labels import loader as label_sets
from engine.repositories.dsld.raw import loader as raw
from engine.services import dsld_cleaning

N = 100
SEED = 20260930

## Get

A seeded random draw of `N` ids over everything the dump holds.

In [2]:
raw_labels = raw.load_raw(n=N, order="random", seed=SEED)
len(raw_labels)

100

In [3]:
# What a raw record carries -- most of it is not label text.
# sorted(raw_labels[0])
raw_labels[0]



{'src': '01-raw/2025-09-25/data/label_136.json',
 'id': 293979,
 'nhanesId': '',
 'bundleName': '',
 'fullName': 'Earth Broth',
 'brandName': 'HealthForce SuperFoods',
 'brandIpSymbol': '®',
 'upcSku': '6 50786 00347 6',
 'productVersionCode': 'EARTHC120|21081001 Version 5',
 'pdf': '293979_document.pdf',
 'thumbnail': '293979_thumbnail.jpg',
 'servingsPerContainer': '15',
 'hasOuterCarton': None,
 'percentDvFootnote': 'Percent Daily Values are based on a 2,000 calorie diet',
 'labelRelationships': [],
 'contacts': [{'contactId': 25637,
   'text': 'Manufactured by',
   'types': ['Manufacturer'],
   'contactDetails': {'src': '01-raw/2025-09-25/data/contact_5.json',
    'id': 25637,
    'name': 'Excelsior Alchemy',
    'streetAddress': '',
    'city': 'Tempe',
    'state': 'AZ',
    'country': '',
    'zipCode': '85284',
    'phoneNumber': '(800) 748-6229',
    'email': '',
    'webAddress': 'www.healthforcesuperfoods.com'}}],
 'netContents': [{'order': 1,
   'quantity': 120,
   'unit': 

## Clean

Five paths are read: `fullName`, `statements[i].notes`, and
`notes`/`description` on every ingredient row, descending `nestedRows`.
Everything else is absent because nothing reads it.

Then each segment is
- **normalised**: lowercase; quantities out (`500 mg`, `2-3 capsules`, `10%`,
  a bare `2`), but names like `b12`, `omega-3` and age limits like
  `under the age of 18` kept; punctuation to spaces; whitespace collapsed;
- **stripped** of every phrase on the checklist,
  `engine/services/dsld_cleaning/boilerplate.txt`, except in `fullName`. Only
  the phrase goes, so a mixed line keeps its real caution;
- **dropped** if nothing is left.

In [4]:
cleaned = dsld_cleaning.clean(raw_labels)
cleaned[0]

CleanedLabel(dsld_id=293979, name='Earth Broth', segments=[Segment(text='Earth Broth', paths=['fullName']), Segment(text='Earth Broth is a one-of-a-kind superfood complex for deep nourishment and adaptogenic support. The perfect companion product to Vitamineral Green, earth is warming, grounding, and savory.', paths=['statements[0].notes']), Segment(text='Earth broth is abundant in nutrient-dense roots such as astragalus, burdock, yacon, and ashwagandha as well as beta- glucan endowed red reishi, shiitake, and maitake mushroom extracts. Earth broth also contains a diverse mix of beneficial herbs, including slippery elm bark, red clover, kelp, blessed thistle, and sheep sorrel.', paths=['statements[1].notes']), Segment(text='Suggested use: Start with 2 VeganCaps per day and increase gradually over two weeks to suggested use of 8 VeganCaps per day, or as advised by a qualified health care professional. Earth broth capsules are convenient for traveling and when on the go.', paths=['statem

In [5]:
# Each surviving piece of text remembers every place it came from.
for segment in cleaned[0].segments[:5]:
    print(segment.paths, segment.text[:70])

['fullName'] Earth Broth
['statements[0].notes'] Earth Broth is a one-of-a-kind superfood complex for deep nourishment 
['statements[1].notes'] Earth broth is abundant in nutrient-dense roots such as astragalus, bu
['statements[2].notes'] Suggested use: Start with 2 VeganCaps per day and increase gradually o
['statements[3].notes'] Refrigerate after opening.


In [ ]:
# Each raw statement next to what v4 kept of it; `-` means it was all
# boilerplate and the segment was dropped.
kept = {p: s.text for s in cleaned[0].segments for p in s.paths}
for i, statement in enumerate(raw_labels[0].get("statements") or []):
    print(f"{str(statement.get('notes'))[:70]!r}\n"
          f"  -> {kept.get(f'statements[{i}].notes', '-')[:70]!r}")

In [ ]:
# Text that is identical after normalising merges, keeping every path: under
# v4 that includes lines that differed only in case, punctuation or an amount.
multi = [s for label in cleaned for s in label.segments if len(s.paths) > 1]
multi[:3]

## Flatten

One row per label. `passage` is what a prompt sends; `segments` rides along as
provenance, and `renderer_version` says which renderer produced the text.

In [7]:
rows = dsld_cleaning.flatten(cleaned)
{k: v for k, v in rows[0].items() if k != "segments"}

{'dsld_id': 293979,
 'name': 'Earth Broth',
 'passage': 'Earth Broth\nEarth Broth is a one-of-a-kind superfood complex for deep nourishment and adaptogenic support. The perfect companion product to Vitamineral Green, earth is warming, grounding, and savory.\nEarth broth is abundant in nutrient-dense roots such as astragalus, burdock, yacon, and ashwagandha as well as beta- glucan endowed red reishi, shiitake, and maitake mushroom extracts. Earth broth also contains a diverse mix of beneficial herbs, including slippery elm bark, red clover, kelp, blessed thistle, and sheep sorrel.\nSuggested use: Start with 2 VeganCaps per day and increase gradually over two weeks to suggested use of 8 VeganCaps per day, or as advised by a qualified health care professional. Earth broth capsules are convenient for traveling and when on the go.\nRefrigerate after opening.\nRoots Seeds Adaptogens Mushrooms Herbs\nPerfect companion to Vitamineral Green\nGrounding Warming Nourishing Comprehensive plant sour

In [8]:
print(rows[0]["passage"][:600])

Earth Broth
Earth Broth is a one-of-a-kind superfood complex for deep nourishment and adaptogenic support. The perfect companion product to Vitamineral Green, earth is warming, grounding, and savory.
Earth broth is abundant in nutrient-dense roots such as astragalus, burdock, yacon, and ashwagandha as well as beta- glucan endowed red reishi, shiitake, and maitake mushroom extracts. Earth broth also contains a diverse mix of beneficial herbs, including slippery elm bark, red clover, kelp, blessed thistle, and sheep sorrel.
Suggested use: Start with 2 VeganCaps per day and increase gradually ove


In [ ]:
# A label whose every segment was boilerplate keeps its row with an empty
# passage, so every raw record still has exactly one row.
empty = [r["dsld_id"] for r in rows if not r["passage"]]
len(empty), empty[:10]

In [ ]:
# Since renderer 3 the product name is the passage's first segment: a name like
# "Sleep Aid" is itself a claim, so a model must be able to quote it. Its
# `fullName` path keeps name-grounded claims distinguishable downstream.
# Under v4 the segment is normalised but never boilerplate-stripped, while the
# row's `name` stays raw: it is provenance, not model input.
rows[0]["name"], rows[0]["segments"][0]

## Save

A name is one set of passages for good: rerunning this cell with the same
rows writes nothing, and saving different rows under a used name is refused.
The name carries the draw and the renderer, so a new seed or renderer gets
a new set.

In [11]:
name = f"random{N}-seed{SEED}-r{dsld_cleaning.RENDERER_VERSION}"
passages_sha = dsld_cleaning.DsldCleaningService().save(rows, name)
name, passages_sha

('random100-seed20260930-r3', 'dd9f20527520e932')

In [ ]:
# Read it back the way a run will.
loaded_sha, labels = label_sets.load_flattened(name)
loaded_sha == passages_sha, len(labels)

## How much the cleaning removes

Characters of raw JSON against characters of passage, over the drawn labels.

In [13]:
import json

raw_chars = sum(len(json.dumps(r)) for r in raw_labels)
passage_chars = sum(len(r["passage"]) for r in rows)
print(f"raw {raw_chars:,} -> passage {passage_chars:,} "
      f"({100 * (1 - passage_chars / raw_chars):.1f}% smaller)")

raw 874,241 -> passage 129,907 (85.1% smaller)
